# Student performance analysis using pandas

This project explores a student performance dataset using pandas.  
The analysis focuses on data cleaning, feature engineering, and exploratory data analysis in order to identify patterns in grades, attendance, pass rates, and academic risk across courses, teachers, cities, and semesters.

## 1. Import libraries and load dataset

In [53]:
import pandas as pd

# Load raw dataset
df = pd.read_csv("data/student_performance_raw.csv")

## 2. Initial data inspection

This section provides a first overview of the dataset, including its size, columns, data types, and summary statistics.

In [56]:
df.head()
df.shape
df.columns
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             61 non-null     int64  
 1   student_name           61 non-null     object 
 2   course                 61 non-null     object 
 3   semester               61 non-null     int64  
 4   teacher                61 non-null     object 
 5   city                   61 non-null     object 
 6   hours_study            60 non-null     float64
 7   absences               60 non-null     float64
 8   assignments_completed  60 non-null     float64
 9   exam_score             60 non-null     float64
 10  final_grade            60 non-null     float64
dtypes: float64(5), int64(2), object(4)
memory usage: 5.4+ KB


,student_id,semester,hours_study,absences,assignments_completed,exam_score,final_grade
count,61.000000,61.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,30.983607,1.508197,6.450000,3.616667,7.616667,74.666667,14.933333
std,17.725210,0.504082,2.086478,2.565030,1.718214,13.418513,2.990758
min,1.000000,1.000000,2.000000,0.000000,4.000000,43.000000,8.000000
25%,16.000000,1.000000,5.000000,2.000000,6.750000,65.750000,13.000000
50%,31.000000,2.000000,7.000000,3.000000,8.000000,77.000000,15.000000
75%,46.000000,2.000000,8.000000,5.000000,9.000000,85.250000,17.000000
max,60.000000,2.000000,10.000000,10.000000,10.000000,96.000000,20.000000


## 3. Data quality checks

This section checks for missing values, duplicate rows, and inconsistent categorical labels.

In [59]:
df.isna().sum()
df.duplicated().sum()
df["course"].unique()
df["city"].unique()
df["teacher"].unique()
df["student_name"].unique()

array(['Nikos', 'Maria', 'Kostas', 'Anna', 'Eleni', 'Giannis', 'Sofia',
       'Petros', 'Dimitra', 'Andreas', 'Vasilis', 'Katerina', 'Rania',
       'Marios', 'Iro', 'Christos', 'Lena', 'Spyros', 'Mina', 'Thanos',
       'Zoe', 'Aris', 'Nefeli', 'Panos', 'Eva', 'Orestis', 'Iason',
       'Dora', 'Melina', 'Fivos', 'Stella', 'Manos', 'Nadia', 'Agapi',
       'Tasos', 'Irini'], dtype=object)

## 4. Data cleaning 
Duplicate rows are removed, text fields are standarised, and missing numeric values are filled using the median

In [62]:
df = df.drop_duplicates()
df["course"] = df["course"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.title()
df["teacher"] = df["teacher"].str.strip().str.title()
df["student_name"] = df["student_name"].str.strip().str.title()
numeric_cols = ["hours_study", "absences", "assignments_completed", "exam_score", "final_grade"]
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
df["assignments_completed"] = df["assignments_completed"].astype(int)
df["absences"] = df["absences"].astype(int)
df = df.reset_index(drop=True)

# Save cleaned Dataset
df.to_csv("data/student_performance_clean.csv",index=False)

## 5. Feature engineering
New features created to support the analysis, including pass status, study efficiency and academic risk classification 


In [65]:
#create pass/fail indicator 
df["pass"] = df["final_grade"] >= 10
#check the case of zero hours_study
df["hours_study"].min() 
#create efficiency metric 
df["efficiency"] = df["final_grade"] / df["hours_study"]
def classify_risk(row):
    if row["final_grade"] <= 10 or row["absences"] >= 8:
        return "high_risk"
    elif row["final_grade"] <= 13 or row["absences"] >= 5:
        return "medium_risk"
    else:
        return "low_risk"
df["risk"] = df.apply(classify_risk, axis=1)   
df[["final_grade", "hours_study", "pass", "efficiency", "risk"]].head(10)
df["risk"].value_counts()
df["pass"].value_counts()

pass
True     57
False     3
Name: count, dtype: int64

## 6. Exploratory data analysis 
This section uses grouping and aggregation techniques to examine student performance across courses, teachers, cities and semesters


  ### 6.1 Average final grade by course

In [69]:
df.groupby("course")["final_grade"].mean().round(1).sort_values(ascending=False)

course
Informatics    16.8
Physics        15.0
Chemistry      14.8
Math           14.1
Biology        13.9
Name: final_grade, dtype: float64

### 6.2 Pass rate by course

In [72]:
(df.groupby("course")["pass"].mean()*100).round(1).sort_values(ascending=True)

course
Biology         91.7
Chemistry       91.7
Physics         91.7
Informatics    100.0
Math           100.0
Name: pass, dtype: float64

### 6.3 Average absences by city

In [75]:
df.groupby("city")["absences"].mean().round(2).sort_values(ascending=False)

city
Heraklion       4.50
Thessaloniki    4.44
Athens          4.11
Patra           3.13
Larisa          1.89
Name: absences, dtype: float64

### 6.4 Teacher summary

In [78]:
teacher_summary = df.groupby("teacher").agg({
    "final_grade": "mean",
    "pass": "mean",
    "student_id": "count",
    "absences": "mean",
    "efficiency": "mean"
}).round(1)
teacher_summary["pass"] = teacher_summary["pass"]*100
teacher_summary.sort_values(["final_grade","pass"],ascending=[False,False])  

,final_grade,pass,student_id,absences,efficiency
teacher,,,,,
Papanikolaou,17.0,100.0,6,2.7,2.2
Karagianni,16.5,100.0,6,2.5,2.3
Stavrou,15.3,100.0,6,3.2,2.3
Nikolaou,15.2,100.0,6,2.8,2.4
Papadopoulos,14.7,100.0,6,3.7,2.3
Konstantinou,14.7,80.0,6,3.7,2.4
Georgiou,14.5,80.0,6,3.5,2.6
Alexiou,14.2,100.0,6,3.8,2.4
Mavridis,13.7,80.0,6,5.3,3.0


### 6.5 Average Final grade by course and semester

In [81]:
df.groupby(["course","semester"])["final_grade"].mean().round(1)

course       semester
Biology      1           13.7
             2           14.2
Chemistry    1           14.5
             2           15.2
Informatics  1           17.0
             2           16.5
Math         1           14.7
             2           13.5
Physics      1           14.7
             2           15.3
Name: final_grade, dtype: float64

### 6.6 Risk distribution by course

In [84]:
df.groupby(["course", "risk"]).size().unstack(fill_value=0)

risk,high_risk,low_risk,medium_risk
course,,,
Biology,2,5,5
Chemistry,1,9,2
Informatics,0,10,2
Math,1,6,5
Physics,1,9,2


### 6.7 Student count and grade by course

In [87]:
df.groupby("course").agg({
    "student_id": "count",
    "final_grade": "mean"
}).round(1).rename(columns={"student_id": "student_count"})

,student_count,final_grade
course,,
Biology,12,13.9
Chemistry,12,14.8
Informatics,12,16.8
Math,12,14.1
Physics,12,15.0


### 6.8 Top-performing students

In [90]:
df.sort_values("final_grade", ascending=False)[["student_name", "course", "final_grade", "absences", "efficiency", "exam_score"]].head(10)

,student_name,course,final_grade,absences,efficiency,exam_score
49,Orestis,Informatics,20.0,0,2.000000,96.0
7,Petros,Math,19.0,0,1.900000,92.0
31,Christos,Biology,19.0,0,2.111111,93.0
39,Petros,Informatics,19.0,1,2.111111,92.0
32,Nikos,Informatics,19.0,1,1.900000,94.0
9,Andreas,Physics,19.0,0,1.900000,95.0
23,Petros,Chemistry,19.0,0,2.111111,91.0
54,Stella,Informatics,19.0,1,2.111111,91.0
51,Dora,Physics,18.0,1,2.250000,88.0
8,Dimitra,Physics,18.0,1,2.000000,90.0


### 6.9 High risk students

In [93]:
high_riskers = df[df["risk"] == "high_risk"]
high_riskers.sort_values(["final_grade","absences"], ascending=[True, False])[[
    "student_name", "final_grade", "course", "exam_score", "efficiency", "risk"
]]


,student_name,final_grade,course,exam_score,efficiency,risk
58,Tasos,8.0,Biology,43.0,4.000000,high_risk
18,Kostas,8.0,Chemistry,45.0,4.000000,high_risk
15,Christos,9.0,Physics,49.0,3.000000,high_risk
2,Kostas,10.0,Math,52.0,3.333333,high_risk
26,Vasilis,10.0,Biology,54.0,3.333333,high_risk


## 7. Key findings

- Biology recorded the lowest average final grade among all courses.
- Mathematics showed a 100% pass rate despite having a relatively lower average final grade.
- Heraklion had the highest average number of absences.
- Teachers with higher average grades also tended to have lower average absences.
- Academic performance showed a slight improvement from semester 1 to semester 2.
- Biology contained the highest number of medium- and high-risk students, while Informatics had the lowest.

## 8. Conclusion

This project demonstrated a complete introductory pandas workflow, including data inspection, cleaning, feature engineering, and grouped exploratory analysis.  
The results show how simple transformations and aggregations can be used to identify meaningful academic performance patterns and student risk categories.